# 05. Logistic Regression: Training-Only Model Selection

Use the existing outer **training** split for all model selection and fitting.
Only section 9 evaluates the saved model on the held-out test split.
The flow is: load training reviews -> join movie features -> grouped CV with fold-local
encoding -> tune all features by mean validation macro-F1 -> backward elimination
-> retune the selected subset -> refit the chosen configuration.

CV selects hyperparameters, not averaged coefficient weights: each fold has its own
vocabulary. Fresh weights are learned during refitting.

## 1. Setup and Search Settings

Initial tuning and backward elimination use a 10,000-review development sample
with five folds. `sample_size` controls only these early stages; set it to `None`
to use the full training partition there too. Sections 7 and 8 always use every
record in the saved training split, regardless of `sample_size`.
Sampling is uniform, without replacement, before grouped CV.

Search L1, L2 and elastic net at four C values, plus one unpenalized model. C has no
effect without regularization. Elastic net's mixing ratio is fixed at 0.5. Compare
unigrams and unigrams + bigrams. The vocabulary cap applies to each text field.
Oversampling is fixed on, for training rows only.

`max_macro_f1_drop = 0.005` allows an absolute 0.5 percentage-point loss from the
best mean macro-F1 on the elimination path. Set this tolerance before inspecting
results. This is a practical trade-off, not a statistical significance test.

In [ ]:
from pathlib import Path
import sys

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "src").is_dir(), "Start in the repository root or notebooks/."
sys.path.insert(0, str(PROJECT_ROOT))

from src.prepare_features import INITIAL_FEATURES, prepare_features
from src.split_data import RANDOM_STATE
from src.logistic_regression import (
    tune_logistic_regression, backward_elimination, refit_logistic_regression,
)
from src.download_data import download_data
from src.data_cleaning import clean_data
from src.split_data import create_train_test_split

sample_size = 10000  # Initial tuning/elimination only; sections 7-8 always use full training.
k = 5
group_by = "review_text"
initial_features = INITIAL_FEATURES.copy()
max_macro_f1_drop = 0.005
oversample_positive = True
max_iter = 1000
tol = 1e-3

model_grid = [
    {
        "penalty": ["l2", "l1", "elasticnet"],
        "C": [0.001, 0.01, 0.1, 1.0]
    },
    {"penalty": [None]}
]

tfidf_grid = {
    "ngram_range": [(1, 1), (1, 2)],
    "min_df": [3],
    "max_features": [5000]
}


## 2. Load Only Training Data

Read the existing training split in chunks to avoid loading the entire file for a
small experiment.

In [ ]:
dataset_files = ["IMDB_reviews.json", "IMDB_movie_details.json"]
raw_dir = PROJECT_ROOT / "data/raw"
cleaned_dir = PROJECT_ROOT / "data/cleaned"

if not all((raw_dir / name).exists() for name in dataset_files):
    download_data()
if not all((cleaned_dir / name).exists() for name in dataset_files):
    clean_data()

split_dir = PROJECT_ROOT / "data/splits"
if group_by == "movie_id":
    split_dir = split_dir / "movie_disjoint"
split_files = ["IMDB_reviews_train.json", "IMDB_reviews_test.json"]
train_path = split_dir / "IMDB_reviews_train.json"
if not all((split_dir / name).exists() for name in split_files):
    create_train_test_split(group_by=group_by)

with train_path.open() as source:
    total_training_rows = sum(1 for _ in source)
selected_rows = None
if sample_size is not None:
    selected_rows = np.random.default_rng(RANDOM_STATE).choice(
        total_training_rows, size=min(sample_size, total_training_rows), replace=False,
    )
with pd.read_json(train_path, lines=True, convert_dates=False, chunksize=10000) as reader:
    reviews = pd.concat([
        chunk if selected_rows is None else chunk.loc[chunk.index.isin(selected_rows)]
        for chunk in reader
    ])
source_row_indices = reviews.index.to_numpy()
reviews = reviews.reset_index(drop=True)

movies = pd.read_json(
    PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
    lines=True, convert_dates=False,
)
model_data = prepare_features(reviews, movies)
model_data["is_spoiler"] = reviews["is_spoiler"].to_numpy()
model_data["movie_id"] = reviews["movie_id"].to_numpy()
print(f"Using {len(model_data):,} of {total_training_rows:,} training reviews.")
print(f"Spoiler prevalence: {model_data['is_spoiler'].mean():.2%}")
del reviews, movies


## 3. Initial Tuning With All Features

Every configuration uses the same grouped folds and original text. Duplicate review
groups stay together. For each TF-IDF configuration, encoders are fitted on that
fold's training rows, then training rows alone are oversampled. The encoded matrices
are reused across logistic regression candidates.

Mean validation **macro-F1** selects the winner, which must converge in every fold. Precision, recall and F1 refer to the spoiler class, using the default 0.5 probability threshold. Macro-F1 and
**average precision (AP)** are also reported.

Means and standard deviations are over validation folds, not training scores.

In [ ]:
baseline_summary, baseline_fold_scores, baseline_params = tune_logistic_regression(
    model_data, model_grid=model_grid, tfidf_grid=tfidf_grid,
    feature_columns=initial_features, k=k, group_by=group_by,
    random_state=RANDOM_STATE, oversample_positive=oversample_positive,
    max_iter=max_iter, tol=tol,
)
display(baseline_summary)
display(baseline_fold_scores.groupby("candidate")[["iterations", "seconds"]].max())
print("All-feature parameters:", baseline_params)


## 4. Inspect the All-Feature Baseline

Unconverged candidates remain visible but are excluded from selection. Increase
`max_iter` and rerun the same grid to compare them properly.

Here, we examine the best hyperparameters for all-feature baseline.

In [ ]:
if baseline_params is None:
    raise RuntimeError("No candidate converged in every fold. Increase max_iter and rerun.")
best_candidate = baseline_summary.index[baseline_summary["converged"]][0]
display(baseline_summary.loc[[best_candidate]])
display(baseline_fold_scores.loc[baseline_fold_scores["candidate"] == best_candidate])


## 5. Backward Feature Elimination

Hold the initial best classifier and TF-IDF hyperparameters fixed. At each step,
try removing each remaining feature block and follow the removal with the highest
mean validation macro-F1. Encoders are fitted once per fold per step, candidate
subsets reuse the sparse matrix by slicing independent feature blocks.

Continue the greedy path down to one usable feature, then choose the **smallest
visited subset within the tolerance of the best path score** (including the baseline).
The tolerance is not applied separately to each removal, which would allow losses
to accumulate. This is a greedy search, not a guarantee of the globally best subset.

The original `model_data` stays intact, even if `review_text` or other fields are
eliminated from model inputs. This keeps validation rows identical
across sample-based baseline tuning and elimination. Section 7 uses the same
grouping strategy on the full training partition, so its folds differ.
Validation is never oversampled.

In [ ]:
selected_features, elimination_path, elimination_trials = backward_elimination(
    model_data, baseline_params, feature_columns=initial_features,
    max_macro_f1_drop=max_macro_f1_drop, k=k, group_by=group_by,
    random_state=RANDOM_STATE, oversample_positive=oversample_positive,
    max_iter=max_iter, tol=tol,
)
print("Selected features:", selected_features)
print("Removed features:", [name for name in initial_features if name not in selected_features])
display(elimination_path)
display(elimination_trials)


## 6. Elimination Plot

The curve follows the best removal at each step, including steps beyond the selected
subset to reveal the performance trade-off. The shaded band is one fold standard
deviation, not a confidence interval. The dashed line marks the allowed score floor;
the highlighted point is the smallest acceptable subset, not necessarily the peak.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
steps = elimination_path["step"].to_numpy()
means = elimination_path["macro_f1_mean"].to_numpy()
stds = elimination_path["macro_f1_std"].to_numpy()
floor = means.max() - max_macro_f1_drop
chosen = elimination_path.loc[elimination_path["selected"]].iloc[0]

ax.plot(steps, means, marker="o", color="tab:blue", label="Greedy elimination path")
ax.fill_between(steps, means - stds, means + stds, color="tab:blue", alpha=0.12,
                label="Fold standard deviation")
ax.axhline(floor, color="tab:green", linestyle="--", label="Allowed score floor")
ax.scatter(chosen["step"], chosen["macro_f1_mean"], color="tab:red", s=130,
           zorder=3, label="Selected subset")
ax.set_xticks(steps, [
    f"{row.n_features} features\n" + (
        "All features" if row.step == 0 else f"- {row.removed_feature}"
    )
    for row in elimination_path.itertuples()
], rotation=30, ha="right")
ax.set_xlabel("Remaining feature blocks and removal at each step")
ax.set_ylabel("Mean validation macro-F1")
ax.set_title("Logistic regression: backward feature elimination")
ax.grid(axis="y", alpha=0.25)
ax.legend(loc="best")
fig.tight_layout()
plt.show()


## 7. Retune With the Selected Features on Full Training Data

Load every record from the saved **training split**, not the combined raw dataset
and not the test split. Keep the selected features fixed, do not rerun elimination.
The data is prepared in chunks and retains only selected inputs, original review
text for grouping, movie IDs, and labels.

Repeat the hyperparameter grid using grouped CV on this full training dataframe.
Macro-F1 remains the selection metric.

The comparison labels each stage's data scope and row count. Differences between
sample and full-training scores are not a paired estimate of feature-selection
improvement. Feature selection and tuning scores are not independent performance
estimates. Full-training CV will require considerably more time and memory.

In [ ]:
if len(model_data) == total_training_rows:
    full_model_data = model_data
else:
    movies = pd.read_json(
        PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
        lines=True, convert_dates=False,
    )
    keep_columns = list(dict.fromkeys(selected_features + ["review_text", "movie_id", "is_spoiler"]))
    prepared_chunks = []
    with pd.read_json(train_path, lines=True, convert_dates=False, chunksize=10000) as reader:
        for reviews in reader:
            features = prepare_features(reviews, movies)
            features["is_spoiler"] = reviews["is_spoiler"].to_numpy()
            features["movie_id"] = reviews["movie_id"].to_numpy()
            prepared_chunks.append(features[keep_columns])
    full_model_data = pd.concat(prepared_chunks, ignore_index=True)
    del prepared_chunks, reviews, features, movies

print(f"Final tuning and refitting: {len(full_model_data):,} training reviews.")
print("Selected features:", selected_features)

In [ ]:
summary, fold_scores, best_params = tune_logistic_regression(
    full_model_data, model_grid=model_grid, tfidf_grid=tfidf_grid,
    feature_columns=selected_features, k=k, group_by=group_by,
    random_state=RANDOM_STATE, oversample_positive=oversample_positive,
    max_iter=max_iter, tol=tol,
)
if best_params is None:
    raise RuntimeError("No retuned candidate converged in every fold. Increase max_iter and rerun.")
display(summary)
print("Retuned parameters:", best_params)

baseline_best = baseline_summary.loc[baseline_summary["converged"]].iloc[0]
retuned_best = summary.loc[summary["converged"]].iloc[0]
eliminated_best = elimination_path.loc[elimination_path["selected"]].iloc[0]
comparison = pd.DataFrame([
    {"stage": "All features, tuned", "n_features": len(initial_features),
     "data_scope": "Selection data", "n_rows": len(model_data),
     "macro_f1_mean": baseline_best["macro_f1_mean"],
     "macro_f1_std": baseline_best["macro_f1_std"]},
    {"stage": "Selected features, fixed parameters", "n_features": len(selected_features),
     "data_scope": "Selection data", "n_rows": len(model_data),
     "macro_f1_mean": eliminated_best["macro_f1_mean"],
     "macro_f1_std": eliminated_best["macro_f1_std"]},
    {"stage": "Selected features, retuned", "n_features": len(selected_features),
     "data_scope": "Full training", "n_rows": len(full_model_data),
     "macro_f1_mean": retuned_best["macro_f1_mean"],
     "macro_f1_std": retuned_best["macro_f1_std"]},
])
display(comparison)

## 8. Refit and Save the Full-Training Model

Fit fresh encoders on every row of `full_model_data`, using only the selected
features and the hyperparameters chosen in section 7.

The saved pipeline is now trained on the full training partition, even when
initial tuning and feature elimination used 10,000 rows. It includes the fitted
encoders and classifier. Prediction does not oversample.

Both tuning stages, the elimination path, removal trials and plot are saved under
the `full_backward` directory. Save full-training row indices separately from
the sample's selection-row indices so the two stages remain reproducible.

In [ ]:
model = refit_logistic_regression(
    full_model_data, best_params, feature_columns=selected_features,
    random_state=RANDOM_STATE, oversample_positive=oversample_positive,
    max_iter=max_iter, tol=tol,
)

scope = "full"
output_dir = PROJECT_ROOT / "data/models" / f"logistic_regression_{group_by}_{scope}_backward"
output_dir.mkdir(parents=True, exist_ok=True)
baseline_summary.to_csv(output_dir / "baseline_cv_summary.csv")
baseline_fold_scores.to_csv(output_dir / "baseline_cv_fold_scores.csv", index=False)
elimination_path.to_json(output_dir / "elimination_path.json", orient="records", indent=2)
elimination_trials.to_json(output_dir / "elimination_trials.json", orient="records", indent=2)
fig.savefig(output_dir / "backward_elimination.png", dpi=150, bbox_inches="tight")
summary.to_csv(output_dir / "cv_summary.csv")
fold_scores.to_csv(output_dir / "cv_fold_scores.csv", index=False)
np.save(output_dir / "training_row_indices.npy", np.arange(len(full_model_data)))
np.save(output_dir / "selection_row_indices.npy", source_row_indices)
joblib.dump({
    "pipeline": model,
    "best_params": best_params,
    "feature_columns": selected_features,
    "training_path": str(train_path),
    "training_rows": len(full_model_data),
    "feature_selection_rows": len(model_data),
    "total_training_rows": total_training_rows,
    "k": k,
    "group_by": group_by,
    "random_state": RANDOM_STATE,
    "oversample_positive": oversample_positive,
    "remove_spoilers": False,
    "selection_metric": "mean validation macro_f1",
    "initial_features": initial_features,
    "baseline_params": baseline_params,
    "max_macro_f1_drop": max_macro_f1_drop,
    "model_grid": model_grid,
    "tfidf_grid": tfidf_grid,
    "max_iter": max_iter,
    "tol": tol,
    "sklearn_version": sklearn.__version__,
}, output_dir / "model.joblib", compress=3)
print("Saved model and CV results:", output_dir)


## 9. Final Held-Out Test Predictions and Metrics

Run only after the feature set, hyperparameters and decision threshold are frozen.
This section reloads the saved full-training pipeline and uses the corresponding
test split. It can be run after section 1 without repeating training, provided
section 8 has already saved the model for the chosen `group_by` setting.

Test rows are processed in chunks with the existing `prepare_features` helper.
The fitted encoders only transform data, there is no fitting, feature selection, ablation,
oversampling or threshold tuning occurs. Keep the existing 0.5 threshold.

Report macro-F1 as the primary metric, plus spoiler-class precision, recall, F1,
accuracy and average precision.

In [ ]:
from src.metrics import evaluate

evaluation_dir = PROJECT_ROOT / "data/models" / f"logistic_regression_{group_by}_full_backward"
saved_model = joblib.load(evaluation_dir / "model.joblib")
test_model = saved_model["pipeline"]
test_path = Path(saved_model["training_path"]).with_name("IMDB_reviews_test.json")
test_movies = pd.read_json(
    PROJECT_ROOT / "data/cleaned/IMDB_movie_details.json",
    lines=True, convert_dates=False,
)

prediction_chunks = []
with pd.read_json(test_path, lines=True, convert_dates=False, chunksize=10000) as reader:
    for test_reviews in reader:
        test_features = prepare_features(test_reviews, test_movies)[saved_model["feature_columns"]]
        probabilities = test_model.predict_proba(test_features)[:, 1]
        prediction_chunks.append(pd.DataFrame({
            "test_row": test_reviews.index.to_numpy(),
            "movie_id": test_reviews["movie_id"].to_numpy(),
            "is_spoiler": test_reviews["is_spoiler"].to_numpy().astype(int),
            "predicted_spoiler": (probabilities > 0.5).astype(int),
            "spoiler_probability": probabilities,
        }))
test_predictions = pd.concat(prediction_chunks, ignore_index=True)
del prediction_chunks, test_reviews, test_features, test_movies
test_scores = evaluate(
    test_predictions["is_spoiler"],
    test_predictions["predicted_spoiler"],
    test_predictions["spoiler_probability"],
)
test_scores["average_precision"] = test_scores.pop("pr_auc")
test_metrics = pd.DataFrame([test_scores])
metric_order = ["macro_f1", "precision", "recall", "f1", "average_precision", "accuracy"]
test_confusion = pd.DataFrame(
    [[test_scores["tn"], test_scores["fp"]], [test_scores["fn"], test_scores["tp"]]],
    index=["Actual non-spoiler", "Actual spoiler"],
    columns=["Predicted non-spoiler", "Predicted spoiler"],
)
print(f"Held-out test reviews: {len(test_predictions):,}")
display(test_metrics[metric_order].T.rename(columns={0: "Test score"}))
display(test_confusion)
display(test_predictions.head())

test_predictions.to_csv(evaluation_dir / "test_predictions.csv", index=False)
test_metrics.to_csv(evaluation_dir / "test_metrics.csv", index=False)
test_confusion.to_csv(evaluation_dir / "test_confusion_matrix.csv")
print("Saved test predictions and metrics:", evaluation_dir)
